# Import library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.utils.data as data
import torch.backends.cudnn as cudnn
from torch.utils.data import random_split
from torchvision import datasets, transforms
from PIL import Image
import os
import random
import numpy as np

# Dataloader

In [3]:
%mkdir dataset
%cd dataset
!gdown 0B_tExHiYS-0veklUZHFYT19KYjg
!tar -zvxf mnist_m.tar.gz
%cd ..

Streaming output truncated to the last 5000 lines.
mnist_m/mnist_m_test/00004544.png
mnist_m/mnist_m_test/00000258.png
mnist_m/mnist_m_test/00004074.png
mnist_m/mnist_m_test/00006977.png
mnist_m/mnist_m_test/00004200.png
mnist_m/mnist_m_test/00005651.png
mnist_m/mnist_m_test/00008563.png
mnist_m/mnist_m_test/00008032.png
mnist_m/mnist_m_test/00003311.png
mnist_m/mnist_m_test/00003541.png
mnist_m/mnist_m_test/00003759.png
mnist_m/mnist_m_test/00005223.png
mnist_m/mnist_m_test/00008205.png
mnist_m/mnist_m_test/00007685.png
mnist_m/mnist_m_test/00005618.png
mnist_m/mnist_m_test/00001756.png
mnist_m/mnist_m_test/00005528.png
mnist_m/mnist_m_test/00008606.png
mnist_m/mnist_m_test/00005159.png
mnist_m/mnist_m_test/00005128.png
mnist_m/mnist_m_test/00001825.png
mnist_m/mnist_m_test/00001524.png
mnist_m/mnist_m_test/00003892.png
mnist_m/mnist_m_test/00006887.png
mnist_m/mnist_m_test/00007919.png
mnist_m/mnist_m_test/00005374.png
mnist_m/mnist_m_test/00000191.png
mnist_m/mnist_m_test/00001185.p

In [4]:
class GetLoader(data.Dataset):
    def __init__(self, data_root, data_list, transform=None):
        self.root = data_root
        self.transform = transform
        self.img_paths = []
        self.img_labels = []

        # Read file and process image paths and labels
        with open(data_list, 'r') as f:
            for line in f:
                parts = line.strip().split() # split by whitespace
                if len(parts) == 2: # ensure valid format
                    img_path, label = parts
                    self.img_paths.append(img_path)
                    self.img_labels.append(int(label)) # convert label into integer

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        label = self.img_labels[index]

        # load image and apply transformation if needed
        img = Image.open(os.path.join(self.root, img_path)).convert('RGB')
        if self.transform:
            img = self.transform(img)

        return img, label

    def __len__(self):
        return len(self.img_paths)





## Configs

In [5]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True # Optimize perfomance for fixed input sizes

# dataser and model paths
source_dataset_name = 'MNIST'
target_dataset_name = 'mnist_m'

source_image_root = os.path.join('./dataset', source_dataset_name)
target_image_root = os.path.join('./dataset', target_dataset_name)
model_root = './models'
os.makedirs(model_root, exist_ok=True) # Ensure model directory exists

# training hyper parameters
lr = 1e-3
batch_size = 128
image_size = 28
n_epoch = 20
val_split = 0.2
num_workers = 4 # number of paraller data-loading workers

# function to initialize random seeds for reproducibility
def init_seeds(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    cudnn.deterministic = True # ensure deterministic behavior for CUDA operations

init_seeds()

## Prepare data

In [6]:
import torch.utils
import torch.utils.data


img_transform_source = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307), std=(0.3081)) # normalize using MNIST stat
])

img_transform_target = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)) # normalize for target domain
])

# load MINST dataset (source domain)
dataset_source = datasets.MNIST(
    root='../dataset', train=True, transform=img_transform_source, download=True
)

# split source dataset into training and validation sets
num_val = int(val_split * len(dataset_source))
dataset_source_train, dataset_source_val = random_split(dataset_source, [len(dataset_source) - num_val, num_val])

# create dataloaders for source domain
dataloader_source_train = torch.utils.data.DataLoader(
    dataset_source_train, batch_size=batch_size, shuffle=True, num_workers=num_workers
)

dataloader_source_val = torch.utils.data.DataLoader(
    dataset_source_val, batch_size=batch_size, shuffle=False, num_workers=num_workers
)

# load target dataset using custom GetLoader
train_list = os.path.join(target_image_root, 'mnist_m_train_labels.txt')
dataset_target = GetLoader(
    data_root=os.path.join(target_image_root, 'mnist_m_train'),
    data_list=train_list,
    transform=img_transform_target
)

# split target dataset into training and validation sets
num_val = int(val_split * len(dataset_target))
dataset_target_train, dataset_target_val = random_split(dataset_target, [len(dataset_target) - num_val, num_val])

# create dataloader for target domain
dataloader_target_train = torch.utils.data.DataLoader(
    dataset_target_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, persistent_workers=True
)

dataloader_target_val = torch.utils.data.DataLoader(
    dataset_target_val, batch_size=batch_size, shuffle=False, num_workers=num_workers
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 339kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.82MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


# Define model

In [7]:
from torch.autograd import Function

# gradient reversal layer
class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x) # forward pass without modification

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None # reverse gradient during backpropagation

# domain-adversarial neural network(DANN)
class DANN(nn.Module):
    def __init__(self):
        super(DANN, self).__init__()

        # feature extractor
        self.feature = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            nn.ReLU(True),
            nn.Conv2d(64, 50, kernel_size=5),
            nn.BatchNorm2d(50),
            nn.Dropout2d(),
            nn.MaxPool2d(2),
            nn.ReLU(True),
            nn.Flatten()
        )
        # classifier for predicting digit labels (0-9)
        self.class_classifier = nn.Sequential(
            nn.Linear(50 * 4 * 4, 100),
            nn.BatchNorm1d(100),
            nn.ReLU(True),
            nn.Dropout2d(),
            nn.Linear(100, 100),
            nn.BatchNorm1d(100),
            nn.ReLU(True),
            nn.Linear(100, 10),
            nn.LogSoftmax(dim=1)
        )
        # domain classifier for distinguishing source and target domain
        self.domain_classifier = nn.Sequential(
            nn.Linear(50 * 4 * 4, 100),
            nn.BatchNorm1d(100),
            nn.ReLU(True),
            nn.Linear(100, 2), # output: 2 classes (source or target)
            nn.LogSoftmax(dim=1)
        )

    def forward(self, input_data, alpha):
        input_data = input_data.expand(input_data.shape[0], 3, 28, 28) # ensure 3 channels input
        feature = self.feature(input_data) # extract feature
        reverse_feature = ReverseLayerF.apply(feature, alpha) # apply gradient reversal
        class_output = self.class_classifier(feature)
        domain_output = self.domain_classifier(reverse_feature)

        return class_output, domain_output


#Initialize the DANN model
my_net = DANN().to(device)


In [8]:
# set up optimizer and loss function
optimizer = optim.Adam(my_net.parameters(), lr=lr)
loss_class = torch.nn.NLLLoss().to(device) # classification loss
loss_domain = torch.nn.NLLLoss().to(device) # domain adaption loss

## Define evaluation

In [9]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad(): # Disable gradient computation
        for img, label in dataloader:
            img, label = img.to(device), label.to(device)
            output, _ = model(img, alpha = 0) # forward pass with no domain classifier output
            pred = output.argmax(dim=1) # get predicted class
            correct += pred.eq(label).sum().item() # count correct predictions
            total += label.size(0)

    return correct / total

# Train

In [10]:
train_acc_source, val_acc_source = [], []
train_acc_target, val_acc_target = [], []

max_acc = 0.0 # store the best target accuracy
best_model_path = os.path.join(model_root, 'best_model.pth')

for epoch in range (n_epoch):
    my_net.train() # set mode to train
    len_dataloader = min(len(dataloader_source_train), len(dataloader_target_train))
    for i, (data_source_train, data_target_train) in enumerate(zip(dataloader_source_train, dataset_target_train)):
        p = (i + epoch * len_dataloader) / (n_epoch * len_dataloader) # compute training progress
        alpha = 2.0 / (1.0 + np.exp(-10 * p)) - 1 # compute alpha for domain adaptation

        # process source domain data
        s_img, s_label = data_source_train
        s_img, s_label = s_img.to(device), s_label.to(device)
        domain_label_source = torch.zeros(len(s_label), dtype=torch.long, device=device)

        my_net.zero_grad()
        class_output, domain_output = my_net(s_img, alpha)
        err_s_label = loss_class(class_output, s_label) # Classification loss (source)
        err_s_domain = loss_domain(domain_output, domain_label_source) # domain loss (source)

        # process target domain data
        t_img, _ = data_target_train
        t_img = t_img.to(device)
        domain_label_target = torch.ones(len(t_img), dtype=torch.long, device=device)

        _, domain_output = my_net(t_img, alpha)
        err_t_domain = loss_domain(domain_output, domain_label_target) # domain loss (target)

        # compute total loss and update model
        loss = err_s_label + err_s_domain + err_t_domain
        loss.backward()
        optimizer.step()

    # evaluate model on source and target domains
    train_acc_source.append(evaluate(my_net, dataloader_source_train))
    val_acc_source.append(evaluate(my_net, dataloader_source_val))
    train_acc_target.append(evaluate(my_net, dataloader_target_train))
    val_acc_target.append(evaluate(my_net, dataloader_target_val))

    source_acc = val_acc_source[-1]
    target_acc = val_acc_target[-1]
    print(f"Epoch {epoch}: MNIST_acc {source_acc:.4f} mnist_m_acc: {target_acc:.4f}")

    # Save best model based on target domain accuracy
    if target_acc > max_acc:
        max_acc = target_acc
        torch.save(my_net.state_dict(), best_model_path)

/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:1538: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  warnings.warn(warn_msg)


Epoch 0: MNIST_acc 0.9738 mnist_m_acc: 0.2903
Epoch 1: MNIST_acc 0.9818 mnist_m_acc: 0.2923
Epoch 2: MNIST_acc 0.9781 mnist_m_acc: 0.2969
Epoch 3: MNIST_acc 0.9801 mnist_m_acc: 0.2842
Epoch 4: MNIST_acc 0.9749 mnist_m_acc: 0.2837
Epoch 5: MNIST_acc 0.9787 mnist_m_acc: 0.2548
Epoch 6: MNIST_acc 0.9778 mnist_m_acc: 0.2697
Epoch 7: MNIST_acc 0.9809 mnist_m_acc: 0.2450
Epoch 8: MNIST_acc 0.9822 mnist_m_acc: 0.3175
Epoch 9: MNIST_acc 0.9822 mnist_m_acc: 0.2874
Epoch 10: MNIST_acc 0.9827 mnist_m_acc: 0.2696
Epoch 11: MNIST_acc 0.9824 mnist_m_acc: 0.3347
Epoch 12: MNIST_acc 0.9847 mnist_m_acc: 0.3186
Epoch 13: MNIST_acc 0.9838 mnist_m_acc: 0.3011
Epoch 14: MNIST_acc 0.9850 mnist_m_acc: 0.2893
Epoch 15: MNIST_acc 0.9841 mnist_m_acc: 0.2801
Epoch 16: MNIST_acc 0.9763 mnist_m_acc: 0.2447
Epoch 17: MNIST_acc 0.9769 mnist_m_acc: 0.2317
Epoch 18: MNIST_acc 0.9794 mnist_m_acc: 0.2788
Epoch 19: MNIST_acc 0.9810 mnist_m_acc: 0.2706


# Evaluate

In [ ]:
data = [train_acc_source, val_acc_source, train_acc_target, val_acc_target]
titles = ['Train Acc Source', 'Val Acc Source', 'Train Acc Target', 'Val Acc Target']
colors = ['b', 'r', 'g', 'm']

# Create a 2x2 subplot for visualization
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, d, title, color in zip(axes.flat, data, titles, colors):
    ax.plot(d, color=color, label=title)  # Plot accuracy curve
    ax.set_title(title)  # Set plot title
    ax.set_xlabel('Epoch')  # Label x-axis
    ax.set_ylabel('Accuracy')  # Label y-axis
    ax.legend()

plt.tight_layout()
plt.show()
